In [1]:
from booknlp.booknlp import BookNLP
from allennlp.predictors.predictor import Predictor
import pandas as pd

import utils
import spacy
nlp = spacy.load("en_core_web_sm")
import pandas as pd
import os
import glob
import json
from collections import Counter
import re

/Users/aidan/Desktop/Work/postdoc/act_textanalysis/SRL3/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


using device cpu


In [2]:
#from allennlp.predictors.predictor import Predictor
SRL_MODEL_PATH = "srl.tar.gz"
predictor_srl = Predictor.from_path(SRL_MODEL_PATH)

In [3]:
model_params={
		"pipeline":"entity,quote,supersense,event,coref", 
		"model":"big"
	}
	
booknlp=BookNLP("en", model_params)

{'pipeline': 'entity,quote,supersense,event,coref', 'model': 'big'}
--- startup: 9.232 seconds ---


In [4]:
def clean_gutenberg_text(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as f:
        text = f.read()

    # 2. Remove page numbers or headers like [Page 1]
    text = re.sub(r'\[Page \d+\]', '', text)

    # 3. Normalize whitespace
    # Remove extra spaces
    text = re.sub(r'[ \t]+', ' ', text)
    # Merge broken lines (assume that a line ending with a lowercase letter continues)
    text = re.sub(r'(?<=[a-z,;])\n(?=[a-zA-Z])', ' ', text)
    # Normalize multiple blank lines to exactly two (paragraph break)
    text = re.sub(r'\n{2,}', '\n\n', text)
    # Strip leading/trailing whitespace
    text = text.strip()

    # 4. Fix Unicode characters (basic replacements)
    text = text.replace('“', '"').replace('”', '"').replace('‘', "'").replace('’', "'")
    text = text.replace('—', '-').replace('–', '-')  # em dash and en dash to hyphen
    text = text.replace('\r', '')  # Remove carriage returns if any
    text = text.replace('_','')

    # 6. Write cleaned text
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(text)

    print(f"Cleaned text saved to {output_path}")

In [5]:
def proc(filename):
    with open(filename) as file:
        data=json.load(file)
    return data

def read_files_in(output_directory, book_id):
    df_tokens = pd.read_csv("./output_dir/" + book_id + "/" + book_id + ".tokens", delimiter="\t")
    df_entities = pd.read_csv("./output_dir/" + book_id + "/" + book_id + ".entities", delimiter="\t")
    data=proc(output_directory + "/" + book_id + ".book")

    return df_tokens, df_entities, data

In [6]:
def get_counter_from_dependency_list(dep_list):
    counter=Counter()
    for token in dep_list:
        term=token["w"]
        tokenGlobalIndex=token["i"]
        counter[term]+=1
    return counter

def get_mods(data, df_tokens, output_directory, book_id):
    mod_rows = []

    for character in data["characters"]:

        agentList=character["agent"]
        patientList=character["patient"]
        possList=character["poss"]
        modList=character["mod"]

        character_id=character["id"]
        count=character["count"]

        referential_gender_distribution=referential_gender_prediction="unknown"

        if character["g"] is not None and character["g"] != "unknown":
            referential_gender_distribution=character["g"]["inference"]
            referential_gender=character["g"]["argmax"]

        mentions=character["mentions"]
        proper_mentions=mentions["proper"]
        max_proper_mention=""

        if len(mentions["proper"]) > 0:
            max_proper_mention=mentions["proper"][0]["n"]

            for m in modList:
                mod_rows.append({
                    "character_id": character_id,
                    'character_name': max_proper_mention,
                    'character_gender': referential_gender,
                    "modifier": m['w'],
                    "modifier_tokenid": m['i'],
                    "sentence_id": df_tokens[df_tokens['token_ID_within_document'] == m['i']]['sentence_ID'].iloc[0],
                    "modifier_tokenidinsentence": df_tokens[df_tokens['token_ID_within_document'] == m['i']]['token_ID_within_sentence'].iloc[0]
                })

        # # just print out information about named characters
        # if len(mentions["proper"]) > 0:
        #     print(character_id, count, max_proper_mention, referential_gender)

        #     print()
        #     printTop=3
            # for k, v in get_counter_from_dependency_list(possList).most_common(printTop):
            #     print("\tposs\t%s %s" % (v,k))
            # print()
            # for k, v in get_counter_from_dependency_list(agentList).most_common(printTop):
            #     print("\tagent\t%s %s" % (v,k))
            # print()
            # for k, v in get_counter_from_dependency_list(patientList).most_common(printTop):
            #     print("\tpatient\t%s %s" % (v,k))
            # print()
            # for k, v in get_counter_from_dependency_list(modList).most_common(printTop):
            #     print("\tmod\t%s %s" % (v,k))
            # print()

    mod_df = pd.DataFrame(mod_rows)
    mod_df.to_csv(output_directory + "/" + book_id + "__" + "modlist.csv")

In [7]:
def check_for_arg(n, arg_name, dict_name, sentence_ID, tokens_df):
    if arg_name in n["tags"]:
        index = n['tags'].index(arg_name)
        if str(sentence_ID)+":"+str(index) in dict_name.keys():
            # the interlocutor is another named character
            return dict_name[str(sentence_ID)+":"+str(index)], 'character_triple'
        else:
            # the interlocutor is not another named character
            # but note, most of these are really bipartite events that happen to have a stop word in the other position
            # return None, None
            return tokens_df[(tokens_df['sentence_ID'] == sentence_ID) & (tokens_df['token_ID_within_sentence'] == index)]['word'].iloc[0], 'other_triple'
    else:
        return None, None

In [8]:
def make_chardict(data, df_tokens):
    #create dictionary with the shape [sentence_ID][token_ID_within_sentence][word]
    sentence_and_token_ID_to_word_dict = {}
    for character in data["characters"]:
        character_id=character["id"]
        mentions=character["mentions"]
        proper_mentions=mentions["proper"]
        if len(mentions["proper"]) > 0:
            max_proper_mention=mentions["proper"][0]["n"]
        else:
            max_proper_mention = character_id

        unique_start_token_ids = df_entities.loc[df_entities['COREF'] == character_id, 'start_token']
        #print(unique_start_token_ids)
        sentence_ids = df_tokens[df_tokens['token_ID_within_document'].isin(unique_start_token_ids)][['token_ID_within_document', 'token_ID_within_sentence', 'sentence_ID']]
        sentence_ids = sentence_ids.reset_index(drop=True)

        for index, row in sentence_ids.iterrows():
            sentence_ID = row['sentence_ID']
            # token_ID_doc = row['token_ID_within_document']
            token_ID_sent = row['token_ID_within_sentence']
            combined_sentence_and_token_ID = str(sentence_ID) + ":" + str(token_ID_sent)
            sentence_and_token_ID_to_word_dict[combined_sentence_and_token_ID] = max_proper_mention

    return sentence_and_token_ID_to_word_dict

In [9]:
def get_events(data, nTop, df_entities, df_tokens, output_directory, book_id):
    # for each of the top X characters, get the sentences they are involved in, find the events
    # this is slow--11 minutes for Winnie the Pooh

    sentence_and_token_ID_to_word_dict = make_chardict(data, df_tokens)

    # nTop = 10
    topChars = sorted(
        [char for char in data['characters'] if len(char['mentions']['proper']) > 0],
        key=lambda x: x['count'],
        reverse=True
    )[:nTop]

    columns = ['character_name', 'character_id', 'sentence_ID', 'character_doctokenid', 'verb', 'agent', 'patient']
    # protag = 'Owl'

    for character in topChars:
        srl_results_df = pd.DataFrame(columns=columns)
        character_id = character['id']
        protag = character['mentions']["proper"][0]["n"]
        print(protag)
        # print(character_id)

        #get all sentencs where COREF = character_id
        unique_start_token_ids = df_entities.loc[df_entities['COREF'] == character_id, 'start_token']
        # print(unique_start_token_ids)
        sentence_ids = df_tokens[df_tokens['token_ID_within_document'].isin(unique_start_token_ids)][['token_ID_within_document', 'token_ID_within_sentence', 'sentence_ID']]
        sentence_ids = sentence_ids.reset_index(drop=True)
        # print(sentence_ids)

        # for each mention in our coreference cluster do the following
        for index, row in sentence_ids.iterrows():
            token_ID_within_document = row['token_ID_within_document']
            #word = df_tokens.loc[df_tokens['token_ID_within_document'] == token_ID_within_document, 'word'].values[0]

            sentence_ID = df_tokens.loc[df_tokens['token_ID_within_document'] == token_ID_within_document, 'sentence_ID'].values[0]
            words_list = df_tokens.loc[df_tokens['sentence_ID'] == sentence_ID, 'word'].tolist()
            token_ID_within_sentence = row['token_ID_within_sentence']

            try:
                sentence_for_srl = ' '.join(words_list)
            except:
                print("Error: ", words_list)
                continue
            predictions_srl = predictor_srl.predict(sentence_for_srl)
            # print(predictions_srl)

            for n in predictions_srl['verbs']:
                if token_ID_within_sentence < len(n['tags']):
                    tag = n['tags'][token_ID_within_sentence]
                    if "B-ARG0" in tag:
                        word, event_type = check_for_arg(n, "B-ARG1", sentence_and_token_ID_to_word_dict, sentence_ID, df_tokens)
                        if word == None: event_type = 'agent_bipart'
                        # if word != None: # getting the two-part events too
                        new_row = pd.DataFrame({'character_name': protag,
                                                'character_id': character_id,
                                                'sentence_ID': [sentence_ID], 
                                                'character_doctokenid': token_ID_within_document,
                                                'verb': [utils.lemmatize(n['verb'], nlp)], 
                                                'agent': [protag], 
                                                'patient': [word],
                                                'event_type': [event_type]})
                        srl_results_df = pd.concat([srl_results_df, new_row], ignore_index=True)

                    elif "B-ARG1" in tag:
                        word, event_type = check_for_arg(n, "B-ARG0", sentence_and_token_ID_to_word_dict, sentence_ID, df_tokens)
                        if word == None: event_type = 'patient_bipart'
                        # if word != None:
                        new_row = pd.DataFrame({'character_name': protag,
                                                'character_id': character_id,
                                                'sentence_ID': [sentence_ID], 
                                                'character_doctokenid': token_ID_within_document,
                                                'verb': [utils.lemmatize(n['verb'], nlp)], 
                                                'agent': [word], 
                                                'patient': [protag],
                                                'event_type': [event_type]})
                        srl_results_df = pd.concat([srl_results_df, new_row], ignore_index=True)
        srl_results_df.to_csv(output_directory + "/events/" + book_id + '__events__' + str(character_id) + ".csv", index=False)
        del srl_results_df

In [10]:
def combine_events(output_directory, book_id):
    # Set the path to your folder containing the CSV files
    folder_path = output_directory + "/events/"  

    # Get a list of all CSV files in the folder
    csv_files = glob.glob(os.path.join(folder_path, '*.csv'))

    # Read and concatenate all CSV files
    combined_df = pd.concat((pd.read_csv(f) for f in csv_files), ignore_index=True)

    # Remove duplicate rows
    combined_df.drop_duplicates(inplace=True)

    # Sort by 'sentence_id'
    combined_df.sort_values(by='sentence_ID', inplace=True)

    # Save the combined, de-duplicated, and sorted DataFrame to a new CSV
    output_path = output_directory + "/" + book_id + "__events.csv"
    combined_df.to_csv(output_path, index=False)

    print(f"Combined CSV saved to {output_path}")

In [11]:
# import pandas as pd
# import os
# import glob
# import re

# # Set the path to your folder containing the CSV files
# folder_path = 'SRL_out_little_princess'  # <-- change this

# # Get a list of all CSV files in the folder
# csv_files = glob.glob(os.path.join(folder_path, '*.csv'))

# # List to collect all processed DataFrames
# dataframes = []

# for file in csv_files:
#     filename = os.path.basename(file)
    
#     # Extract protagonist name using regex
#     match = re.search(r'_output_(.+?)_little_princess\.csv$', filename)
#     if not match:
#         print(f"Could not extract protagonist from filename: {filename}")
#         continue
#     # Replace underscores with spaces and capitalize each word
#     protagonist = match.group(1).replace('_', ' ').title()

#     # Read CSV
#     df = pd.read_csv(file)

#     # Replace 'protag' in all columns (or limit to one if needed)
#     df = df.replace('protag', protagonist)

#     # Add to list
#     dataframes.append(df)

# # Combine all DataFrames
# combined_df = pd.concat(dataframes, ignore_index=True)

# # Remove duplicates
# combined_df.drop_duplicates(inplace=True)

# # Sort by 'sentence_id'
# combined_df.sort_values(by='sentence_ID', inplace=True)

# # Save to CSV
# output_path = os.path.join(folder_path, 'combined_sorted.csv')
# combined_df.to_csv(output_path, index=False)

# print(f"Combined CSV saved to {output_path}")

In [ ]:
# File within this directory will be named ${book_id}.entities, ${book_id}.tokens, etc.
# book_id="winnie_the_pooh"
book_id = "a_little_princess"
nChars = 10

clean_gutenberg_text(
    input_path='data/' + book_id + '.txt',
    output_path='data/' + book_id + '__cleaned.txt',
)

# Input file to process
input_file="data/" + book_id + "__cleaned.txt"

# Output directory to store resulting files in
output_directory='./output_dir/' + book_id

if not os.path.exists(output_directory):
    os.makedirs(output_directory)

if not os.path.exists(output_directory + '/events/'):
    os.makedirs(output_directory + '/events/')

booknlp.process(input_file, output_directory, book_id)

df_tokens, df_entities, data = read_files_in(output_directory, book_id)

get_mods(data, df_tokens, output_directory, book_id)

# this is very slow. 11 minutes for Winnie the Pooh. 20+ for Little Princess.
get_events(data, nChars, df_entities, df_tokens, output_directory, book_id)

combine_events(output_directory, book_id)

Cleaned text saved to data/a_little_princess__cleaned.txt
--- spacy: 9.216 seconds ---
--- entities: 58.220 seconds ---
--- quotes: 0.052 seconds ---
--- attribution: 120.394 seconds ---
--- name coref: 0.066 seconds ---
--- coref: 132.542 seconds ---
--- TOTAL (excl. startup): 320.729 seconds ---, 81690 words
Sara
Miss Minchin
Ermengarde
Becky
